In [5]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from anthropic import Anthropic

In [6]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

In [7]:
response = ollama.chat.completions.create(
    model="gemma3:1b",
    messages=[
        {
            "role": "user",
            "content": "Hello! How are you today?"
        }
    ]
)

print(response.choices[0].message.content)

I'm doing well, thank you for asking! As an AI, I don't experience feelings the way humans do. 😊 But I'm functioning well and ready to help out with any questions you might have. 

And how are **you** today? 

Do you want to talk about something in particular, or perhaps busy with a task?



In [8]:
links = fetch_website_links("https://huggingface.co")
links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/storage',
 '/docs',
 '/enterprise',
 '/pricing',
 '/tasks',
 '/chat',
 '/collections',
 '/languages',
 '/organizations',
 '/blog',
 '/posts',
 '/papers',
 '/hardware',
 '/learn',
 '/join/discord',
 'https://discuss.huggingface.co/',
 'https://github.com/huggingface',
 '/enterprise',
 '/pro',
 '/support',
 '/inference/models',
 '/inference-endpoints',
 '/storage',
 '/login',
 '/join',
 'https://pollen-robotics.com/microduck',
 '/spaces',
 '/models',
 '/Qwen/Qwen3.8-Flash-Next',
 '/zai-org/GLM-5.3-Flash',
 '/zai-org/GLM-5.3',
 '/Qwen/Qwen3.8-27B',
 '/unsloth/Qwen3.8-Flash-Next-GGUF',
 '/models',
 '/spaces/pollen-robotics/microduck-simulator',
 '/spaces/kulkas2pintu/wan555',
 '/spaces/Saravutw/Omni-videos-custom',
 '/spaces/kulkas2pintu/QWEN_EDIT_IMAGE',
 '/spaces/SageBio/rare-disease-real-kid-mva-hackathon-2026',
 '/spaces',
 '/datasets/markov-ai/cad-1000-hours',
 '/datasets/rajpurkar/squad',
 '/datasets/nyu-mll/glue',
 '/datasets/stanfordnl

In [11]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages. 
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [12]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [13]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
https://pollen-robotics.com/microduck
/spaces
/models
/Qwen/Qwen3.8-Flash-Next
/zai-org/GLM-5.3-Flash
/zai-org/GLM-5.3
/Qwen/Qwen3.8-27B
/unsloth/Qwen3.8-Flash-Next-GGUF
/models
/spaces/pollen-robotics/microduck-simulator
/spaces/kulkas2pintu/wan555
/spaces/Saravutw/Omni-videos-custom
/spaces/kulkas2pintu/QWEN_EDIT_IMAGE
/spaces/SageBio/rare-disease-real-kid-mva-hackathon-2026
/spac

In [14]:
import ollama

def select_relevant_links(url):

    response = ollama.chat(
        model="gemma3:1b",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        format="json"
    )

    links = json.loads(response["message"]["content"])
    return links

In [15]:
select_relevant_links("https://huggingface.co")

{'links': ['https://huggingface.co/models',
  'https://huggingface.co/apps',
  'https://huggingface.co/collections/huggingface',
  'https://huggingface.co/conversations',
  'https://huggingface.co/pages/mlflow',
  'https://huggingface.co/pages/transformers',
  'https://huggingface.co/pages/peft',
  'https://huggingface.co/pages/trl',
  'https://huggingface.co/pages/seed',
  'https://huggingface.co/datasets/markov-ai/']}

In [16]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
NEW
Microduck: A Tiny Robot for AI Builders 🦆
Google Gemma 4 is here 💫
Storage Buckets: AI-native object storage
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.8-Flash-Next
Updated
6 days ago
•
208k
•
4.63k
zai-org/GLM-5.3-Flash
Updated
1 day ago
•
441k
•
1.88k
zai-org/GLM-5.3
Updated
1 day ago
•
94.4k
•
1.47k
Qwen/Qwen3.8-27B
Updated
18 days ago
•
4.96M
•
13.6k
unsloth/Qwen3.8-Flash-Next-GGUF
Updated
about 13 hours

In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [19]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [20]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nNEW\nMicroduck: A Tiny Robot for AI Builders 🦆\nGoogle Gemma 4 is here 💫\nStorage Buckets: AI-native object storage\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\

In [ ]:
def create_brochure(company_name, url):
    response = ollama.chat(
        model="gemma3:1b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ]
    )
    return response["message"]["content"]

In [31]:
result = create_brochure("HuggingFace", "https://huggingface.com")
display(Markdown(result))

Okay, here is the brochure text based on the information provided, formatted as you requested:

**Decoding AI – Welcome to Hugging Face**

**(Image: A vibrant collage of AI related images – robots, data points, coding icons, and a friendly face)**

**Are you passionate about making artificial intelligence accessible to everyone?** At Hugging Face, we’re building a thriving community for AI enthusiasts, builders, and researchers. We’re not just a platform; we’re a hub where ideas thrive and AI gets deployed in the real world. 

**About Hugging Face**

We’re the company dedicated to fostering the machine learning community. We offer a collaborative space to:

* **Model & Datasets:** Explore a massive library of models, datasets, and resources – covering a wide range of tasks and languages.
* **Spaces:** Build, share and showcase your AI-powered projects with the world.
* **Organizations:**  Connect with a global network of AI experts & practitioners, ensuring best practices are followed
* **Solutions:** Provide powerful tools for AI development with Inference Endpoints.
* **Explore:** A continuously growing collection of new models and the latest advancements.

**Our Mission:** To help others build better models, more reliably, and with greater support.  We are committed to promoting transparency, sharing code, and accelerating AI innovation.

**For Investors & Partners**

* **Join our thriving ecosystem:** We’re constantly focused on expanding and optimizing Hugging Face’s value with expanding AI research.
*  **Community engagement:** Gain insights through forums, Discord & Blogs.

**For Recruiters**

We are seeking talented individuals to be a valuable asset within Hugging Face. Our career paths include: Data science and engineering and Machine learning engineers

**Learn More**

*   **About Us:** [Link to About Page - 404]  [Page Details]
*   **Careers:**  [Link to Careers Page]
*   **Resources:** [Link to Resources Page]

**---**

Would you like me to revise this, or make some changes?  For example, perhaps we would adjust the visuals based on current branding of Hugging Face?